# CourantMagnetique — `courant_magnetique`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validation of the **differential** post-treatment fields of the standalone
magnetic (induction) equation:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: `courant_magnetique`. Per the
TRUST validation-form layout the other `Postraitement_fields` cases live in
their own sibling folders.

## What it validates

$$
\mathbf{J} = \nabla \times \mathbf{b} / \mu_0
$$
Exactly linear field $b = (y, z, x)$: $\nabla\times b = (-1,-1,-1)$, so with
$\mu_0 = 1$: $J = (-1,-1,-1)$ in every cell. (In 2D $J$ is the single
out-of-plane scalar $J_z$.)


## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | electric current density | $J = \nabla\times b / \mu_0$ |

All these fields are built from the module-reconstructed element gradient
$\nabla b$ (`gradient_b_elements`, Green-Gauss — the kernel has no
element-field gradient factory on PolyMAC). Vector model required: the
scalar model is rejected with a clean error by `creer_champ`.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.



In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

print("BUILD =", BUILD)



## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).



In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "courant_magnetique.data", nbProcs=NB_PROCS)
case.partition()          # trust -partition courant_magnetique.data 2 -> PAR_courant_magnetique.data + DOM_000*.Zones
# --- mesh-convergence scan (shared ladder: L1 = cells x2, L2 = cells x4) ---
# Levels instantiated from src/courant_magnetique.tpl.data (byte-copy of the baseline
# dataset with $nx/$ny/$nz node-count placeholders); the baseline case above is
# the untouched L0. Each level runs in its own build/scan/<lvl>/ directory
# so that the partitioned DOM_000*.Zones never overwrite another's.
NX0 = NY0 = NZ0 = 5                       # baseline node counts (L0)
LEVELS = {
    "L1": {"nx": 2*5-1, "ny": 2*5-1, "nz": 2*5-1},
    "L2": {"nx": 4*5-3, "ny": 4*5-3, "nz": 4*5-3},
}
scan_cases = {}
for lvl, dic in LEVELS.items():
    c = run.addCaseFromTemplate("courant_magnetique.tpl.data", f"scan/{lvl}", dic=dic,
                                targetData=f"courant_magnetique_{lvl}.data",
                                nbProcs=NB_PROCS, excluNR=True)
    c.partition()
    scan_cases[lvl] = c
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)


## Checks


In [ ]:
entries = read_son_all(case.dataFileName_, "SONDE_J")
for name, v in entries:
    print(name, "=", v[-1])
comps = [v[-1] for name, v in entries]

# b = (y, z, x) is exactly linear => curl b = (-1,-1,-1), J = (-1,-1,-1).
expected = np.array([-1., -1., -1.])
print(f"courant_magnetique (probe) : ({comps[0]:.6g}, {comps[1]:.6g}, {comps[2]:.6g})")
err = np.max(np.abs(np.array(comps) - expected)) / np.max(np.abs(expected))
print(f"relative error             : {err:.3e}")
assert err < 1e-12, f"courant_magnetique wrong: {comps}"
print("OK: courant_magnetique = curl b / mu0.")


## Mesh convergence

The shared refinement ladder: `L0` = the baseline mesh above (`5x5x5`),
`L1` = cells x2 (`9x9x9`), `L2` = cells x4 (`17x17x17`).

The field is reconstructed from a linear/uniform `b` at an interior probe,
which is a **discrete identity** (the Green-Gauss reconstruction is exact
for linear fields at interior cells): b = (y,z,x) linear => J = curl b / mu0 = (-1,-1,-1), constant. The scan asserts the same
machine-precision bound at **every** level
(the quantity is position-independent) — no order is reported.

In [ ]:
def read_final_all_dir(subdir, case_name, name):
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    return [np.asarray(son.getValues(e)[1], dtype=float) for e in son.getEntries()]

def level_error(subdir, case_name):
    comps = [float(vv[-1]) for vv in read_final_all_dir(subdir, case_name, "SONDE_J")]
    expected = np.array([-1., -1., -1.])
    e = float(np.max(np.abs(np.array(comps) - expected)))
    return e, expected

scan = {"L0": ("", case.dataFileName_),
        "L1": (f"scan/L1", scan_cases["L1"].dataFileName_),
        "L2": (f"scan/L2", scan_cases["L2"].dataFileName_)}
scan_nodes = {"L0": (5, 5, 5), "L1": (9, 9, 9), "L2": (17, 17, 17)}
res = {lvl: level_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes        abs err        rel")
for lvl in ("L0", "L1", "L2"):
    n0, n1, n2 = scan_nodes[lvl]
    e, expected = res[lvl]
    rel = e / float(np.max(np.abs(expected)))
    print(f"{lvl:<6} {n0:>3}x{n1:>3}x{n2:>3} {e:<13.3e} {rel:.1e}")
    assert rel < 1e-12, f"identity broken at {lvl}: {rel:.3e}"
print("OK: the reconstruction identity holds at every mesh level.")

## Computer Performance


In [ ]:
run.tablePerf()

